Project Written by Ben 

# Home Security System
Automated system to monitor the home security. Teh system handles the following:

**Security Alerts:**
- Motion detected when house is in "away" mode
- Door opened when house is in "away" mode
- Multiple sensors triggered simultaneously (possible break-in)

**Safety Alerts:**
- Temperature below 35°F (frozen pipe risk)
- Temperature above 95°F (equipment failure)
- Smoke detected (fire risk)

**Comfort Notifications:**
- Temperature outside comfort range (65-75°F) when home is in "home" mode
- Unusual patterns (like a door left open for >5 minutes)

## 1. Setup
Imports and the thresholds used by the rules.

In [ ]:

import random
from datetime import datetime, timedelta
from enum import Enum

TEMP_LOW_F = 35         
TEMP_HIGH_F = 95       
TEMP_COMFORT_LOW_F = 65
TEMP_COMFORT_HIGH_F = 75
DOOR_OPEN_LIMIT = timedelta(minutes=5)  

class Mode(Enum):
    HOME = "HOME"
    AWAY = "AWAY"
    SLEEP = "SLEEP"

class SensorType(Enum):
    MOTION = "motion"
    DOOR = "door"
    TEMPERATURE = "temperature"
    SMOKE = "smoke"

class Location(Enum):
    LIVING_ROOM = "Living Room"
    FRONT = "Front"
    KITCHEN = "Kitchen"
    BEDROOM = "Bedroom"

class AlertLevel(Enum):
    CRITICAL = "CRITICAL"
    HIGH = "HIGH"
    INFO = "INFO"
    

class AlertCategory(Enum):
    SECURITY = "SECURITY"
    SAFETY = "SAFETY"
    COMFORT = "COMFORT"


Using enums to define the mode, sensor types, locations, alert levels and alert categories.

# Setting up the sensors 

In [ ]:
def create_sensor(location, sensor_type, value):
    return {"location": location, "type": sensor_type, "current_value": value}

def init_sensors():
    return [
        create_sensor(Location.LIVING_ROOM, SensorType.MOTION, False),
        create_sensor(Location.FRONT, SensorType.DOOR, "CLOSED"),
        create_sensor(Location.KITCHEN, SensorType.TEMPERATURE, 68),
        create_sensor(Location.BEDROOM, SensorType.SMOKE, False),
    ]

## 3.  Implementing Security system


In [ ]:
def create_alert(time, level, category, message):
    return {"time": time, "level": level, "category": category, "message": message}

def check_sensor(sensor, mode, now):
    kind = sensor["type"]
    value = sensor["current_value"]
    message = f"There's a {kind.value} alert at the {sensor['location'].value}"

    # Safety: every mode
    # If this is a smoke sensor and it detects smoke -> critical safety alert (fire risk)
    if kind == SensorType.SMOKE and value:
        return create_alert(now, AlertLevel.CRITICAL, AlertCategory.SAFETY, message + ": fire risk")
    # If this is a temperature sensor and it is below 35°F -> critical safety alert (frozen pipes)
    if kind == SensorType.TEMPERATURE and value < TEMP_LOW_F:
        return create_alert(now, AlertLevel.CRITICAL, AlertCategory.SAFETY, message + ": frozen pipe risk")
    # If this is a temperature sensor and it is above 95°F -> critical safety alert (equipment failure)
    if kind == SensorType.TEMPERATURE and value > TEMP_HIGH_F:
        return create_alert(now, AlertLevel.CRITICAL, AlertCategory.SAFETY, message + ": equipment failure risk")

    # Security: AWAY only
    # If nobody is home (AWAY) and a motion sensor detects movement -> high security alert
    if mode == Mode.AWAY and kind == SensorType.MOTION and value:
        return create_alert(now, AlertLevel.HIGH, AlertCategory.SECURITY, message)
    # If nobody is home (AWAY) and a door is open -> high security alert
    if mode == Mode.AWAY and kind == SensorType.DOOR and value == "OPEN":
        return create_alert(now, AlertLevel.HIGH, AlertCategory.SECURITY, message)

    # Comfort: HOME only
    # If the family is home and it is a temperature sensor...
    if mode == Mode.HOME and kind == SensorType.TEMPERATURE:
        # ...and it is below 65°F or above 75°F -> comfort notice
        if value < TEMP_COMFORT_LOW_F or value > TEMP_COMFORT_HIGH_F:
            return create_alert(now, AlertLevel.INFO, AlertCategory.COMFORT, message)

    # None of the rules matched -> everything is normal, no alert
    return None


## Step 4: Building functions
Generating sensor readings.

In [5]:
def generate_reading(sensor):
    kind = sensor["type"]

    # Motion: about a 15% chance of detecting movement on any reading
    if kind == SensorType.MOTION:
        return random.random() < 0.15

    # Door: about a 10% chance that it is open
    if kind == SensorType.DOOR:
        return "OPEN" if random.random() < 0.10 else "CLOSED"

    # Smoke: rare, about a 2% chance
    if kind == SensorType.SMOKE:
        return random.random() < 0.02

    # Temperature: usually near 68°F, rarely far away from it
    if kind == SensorType.TEMPERATURE:
        return round(random.gauss(68, 12))

In [4]:
for s in init_sensors():
    print(s["type"].value, "->", generate_reading(s))

NameError: name 'init_sensors' is not defined

## 4. Scenario runner

## 5. Fixed test cases
1. AWAY: door opens, then motion
2. AWAY: door + motion in the same tick (possible break-in)
3. SLEEP: temperature boundaries 35/95 and smoke
4. HOME: comfort notice, safety overrides comfort, door open >5 minutes